## Com calculation

In [1]:
#| default_exp mass.com_calculation_module

In [4]:
#| export
import sys
import os
sys.path.append(os.path.abspath('..'))
import numpy as np
from anthromass.mass.measurements_heights_module import *


## Get joint and com postion
Calculates position of COM and Joint centers. Provides these as 2D coordinates. For better understanding of formulas use, take a look at report.

In [ ]:
#| export
def get_joint_and_com_points(Ansur, inputheight, gender):
    m = get_measurements(Ansur, inputheight).iloc[0]
    heights = get_heights(Ansur, inputheight).iloc[0]
    


    #Extract dimensions needed for COM calculations (only for ease of use)
    a1 = m["a1"]
    h1 = m["h1"]
    h2 = m["h2"]
    h3 = m["h3"]
    h4 = m["h4"]
    h6 = m["h6"]
    a5 = m["a5"]
    r1thigh = m["r1thigh"]
    r2thigh = m["r2thigh"]
    # For upper arm, lower arm, etc.
    h8 = m["h8"]
    r1 = m["r1"]
    r2 = m["r2"]
    h10 = m["h10"]
    r3 = m["r3"]
    r4 = m["r4"]

    a2 = m["a2"]
    b2 = m["b2"]
    a3 = m["a3"]
    b3 = m["b3"]
    a4 = m["a4"]

    a7 = m["a7"]
    a8 = m["a8"]
    h9 = m["h9"]
    total_height = heights["TotalH"]





    #Uses (sligthly) coefficinets depending on gender for joint placement

    #Coefficients multiplied with with specific legnth to derive offset  (Based of de leva)
    #Offsets only for vertical direction
    if gender == 1:
        # Calculate offsets
        ShoulderOFF = m["ShoulderJC"] * (-0.10117)
        ElbowOFF    = m["ElbowJC"]    * (-0.04194)
        WristOFF    = m["WristJC"]    * (-0.00597)
        HipOFF      = m["HipJC"]      * (0.00706)
        KneeOFF     = m["KneeJC"]     * (0.00739)
        AnkleOFF    = m["AnkleJC"]    * (-0.03203)
    elif gender == 0:
        ShoulderOFF = m["ShoulderJC"] * (-0.10398)
        ElbowOFF    = m["ElbowJC"]    * (-0.04289)
        WristOFF    = m["WristJC"]    * (-0.00607)
        HipOFF      = m["HipJC"]      * (0.00704)
        KneeOFF     = m["KneeJC"]     * (0.00739)
        AnkleOFF    = m["AnkleJC"]    * (-0.03199)




    ShoulderW = m["acromialW"] #Shoulder joints assumed to be at acromialbreadth
    HipW = m["hipW"]
    Hip_hor = HipW - 0.085 * m["trochanterion-lateralmalleolusheight"] #Special case of Hip offset in horisontal direction
    MidLeg = HipW - r1thigh




    #Calculates each joint position with its accompaning offset
    pointsJC = {
        "Right Shoulder": [ ShoulderW,  m["ShoulderH"] + ShoulderOFF],
        "Shoulder":       [-ShoulderW,  m["ShoulderH"] + ShoulderOFF],
        "Right Elbow":    [ ShoulderW,  m["ElbowH"]    + ElbowOFF],
        "Elbow":          [-ShoulderW,  m["ElbowH"]    + ElbowOFF],
        "Right Wrist":    [ ShoulderW,  m["WristH"]   + WristOFF],
        "Wrist":          [-ShoulderW,  m["WristH"]   + WristOFF],
        "Right Hip":      [ Hip_hor,    m["HipH"]      + HipOFF],
        "Hip":            [-Hip_hor,    m["HipH"]      + HipOFF],
        "Right Knee":     [ MidLeg,     m["KneeH"]     + KneeOFF],
        "Knee":           [-MidLeg,     m["KneeH"]     + KneeOFF],
        "Right Ankle":    [ MidLeg,     m["AnkleH"]    + AnkleOFF],
        "Ankle":          [-MidLeg,     m["AnkleH"]    + AnkleOFF],
    }






    #Calcualtes COM, based on formuals in report. 
    #Defines Position locally (bottom edge at y = 0) (Not actually used)
    #Defines Position in 2D gobally, for (y, z)
    #In some cases height in z direction is calculated first (ends with "z")

    # Upper trunk COM
    UTMCL = [0, h1 / 2] #Local
    UTMC = [0, (h1 / 2 + h2 + h3 + h4 + h6 + a5 * 2)] #Global

    # Head COM
    HeMCL = [0, a7]
    HeMC = [0, (total_height - a7)]

    # Middle trunk COM
    R_top = np.sqrt((a2**2 + b2**2) / 2)
    R_bot = np.sqrt((a3**2 + b3**2) / 2)
    MTMCLz = h2 - (h2 / 4 * ((R_top**2 + 2 * R_top * R_bot + 3 * R_bot**2) /
                              (R_top**2 + R_top * R_bot + R_bot**2)))
    MTMCL = [0, MTMCLz]
    MTMC = [0, (MTMCLz + h3 + h4 + h6 + a5 * 2)]

    # Lower trunk COM
    LTMCL = [0, h3 / 2]
    LTMC = [0, (h3 / 2 + h4 + h6 + a5 * 2)]

    # Left Thigh COM
    LTHMCz = h4 - (h4 * (3 * r2thigh**2 + 2 * r2thigh * r1thigh + r1thigh**2)) / (4 * (r2thigh**2 + r2thigh * r1thigh + r1thigh**2))
    LTHMCL = [-r1thigh, LTHMCz]
    LTHMC = [-MidLeg, LTHMCz + h6 + 2 * a5]

    # Right Thigh COM
    RTHMCz = h4 - (h4 * (3 * r2thigh**2 + 2 * r2thigh * r1thigh + r1thigh**2)) / (4 * (r2thigh**2 + r2thigh * r1thigh + r1thigh**2))
    RTHMCL = [r1thigh, RTHMCz]
    RTHMC = [MidLeg, RTHMCz + h6 + 2 * a5]

    thighdelta = r1thigh - r2thigh

    # Left Shank COM
    LSHMCz = h6 - (h6 * (3 * m["r2shank"]**2 + 2 * m["r2shank"] * m["r1shank"] + m["r1shank"]**2)) / (4 * (m["r2shank"]**2 + m["r2shank"] * m["r1shank"] + m["r1shank"]**2))
    LSHMCL = [0, LSHMCz]
    LSHMC = [-MidLeg, 2 * a5 + LSHMCz]

    # Right Shank COM
    RSHMCz = h6 - (h6 * (3 * m["r2shank"]**2 + 2 * m["r2shank"] * m["r1shank"] + m["r1shank"]**2)) / (4 * (m["r2shank"]**2 + m["r2shank"] * m["r1shank"] + m["r1shank"]**2))
    RSHMCL = [0, RSHMCz]
    RSHMC = [MidLeg, 2 * a5 + RSHMCz]

    # Left Upper Arm COM
    LUAMCz = h8 - (h8 * (3 * r2**2 + 2 * r2 * r1 + r1**2)) / (4 * (r2**2 + r2 * r1 + r1**2))
    LUAMCL = [0, LUAMCz]
    LUAMC = [-ShoulderW, heights["UpperArmH"] + LUAMCz]

    # Right Upper Arm COM
    RUAMCz = h8 - (h8 * (3 * r2**2 + 2 * r2 * r1 + r1**2)) / (4 * (r2**2 + r2 * r1 + r1**2))
    RUAMCL = [0, RUAMCz]
    RUAMC = [ShoulderW, heights["UpperArmH"] + RUAMCz]

    # Left Lower Arm COM
    LLAMCz = h10 - (h10 * (3 * r4**2 + 2 * r4 * r3 + r3**2)) / (4 * (r4**2 + r4 * r3 + r3**2))
    LLAMCL = [0, LLAMCz]
    LLAMC = [-ShoulderW, heights["LowerArmH"] + LLAMCz]

    # Right Lower Arm COM
    RLAMCz = h10 - (h10 * (3 * r4**2 + 2 * r4 * r3 + r3**2)) / (4 * (r4**2 + r4 * r3 + r3**2))
    RLAMCL = [0, RLAMCz]
    RLAMC = [ShoulderW, heights["LowerArmH"] + RLAMCz]

    #Left hand
    LHMCLz = -h9
    LHMCL = [0,LHMCLz]
    LHMC = [-ShoulderW, heights["HandH"]]


    #Right hand
    RHMCLz = -h9
    RHMCL = [0,RHMCLz]
    RHMC = [ShoulderW, heights["HandH"]]


    # Left Foot COM
    LFMCx = m["h7"] - (m["h7"] * (3 * m["a6"]**2 + 2 * m["a6"] * a5 + a5**2)) / (4 * (m["a6"]**2 + m["a6"] * a5 + a5**2))    #Special case since foot is oriandted in a diffrent way
    LFMCL = [LFMCx, a5]
    LFMC = [-MidLeg, a5]

    # Right Foot COM
    RFMCx = m["h7"] - (m["h7"] * (3 * m["a6"]**2 + 2 * m["a6"] * a5 + a5**2)) / (4 * (m["a6"]**2 + m["a6"] * a5 + a5**2))
    RFMCL = [RFMCx, a5]
    RFMC = [MidLeg, a5]

    #Neck
    NMCLz = a8/2
    NMCL = [0,NMCLz]
    NMC = [0, heights["NeckH"]+NMCLz]

    com = {
        "UTMCL": UTMCL, "UTMC": UTMC,
        "HeMCL": HeMCL, "HeMC": HeMC,
        "MTMCL": MTMCL, "MTMC": MTMC,
        "LTMCL": LTMCL, "LTMC": LTMC,
        "LTHMCL": LTHMCL, "LTHMC": LTHMC,
        "RTHMCL": RTHMCL, "RTHMC": RTHMC,
        "LSHMCL": LSHMCL, "LSHMC": LSHMC,
        "RSHMCL": RSHMCL, "RSHMC": RSHMC,
        "LUAMCL": LUAMCL, "LUAMC": LUAMC,
        "RUAMCL": RUAMCL, "RUAMC": RUAMC,
        "LLAMCL": LLAMCL, "LLAMC": LLAMC,
        "RLAMCL": RLAMCL, "RLAMC": RLAMC,
        "LFMCL": LFMCL, "LFMC": LFMC,
        "RFMCL": RFMCL, "RFMC": RFMC, "RFMCx": RFMCx,
        "LHMCL": LHMCL, "LHMC": LHMC,
        "RHMCL": RHMCL, "RHMC": RHMC,
        "NMCL": NMCL, "NMC": NMC
    }




    return pointsJC, com



In [1]:
import nbdev; nbdev.nbdev_export()